# London Flip Finder

**Finding London homes that sold for less than they were worth — and putting a number on how much
confidence that claim deserves.**

A sale record on its own tells you very little. You get the floor area, a few room counts,
whether it was owned outright or leased, the property type, an energy rating, a postal code, and
the price it went for. Nobody values a home from that alone.

What a buyer actually brings to the question is context, and assembling that context is most of
the work here. How far the home is from the nearest metro station, and how central that station
is within the city's transit network. How much crime the neighbourhood recorded in the months
*before* the sale, from official police records — first at a coarse citywide-district level for
the model itself, then re-measured at a much finer neighbourhood level later on, once it came time
to ask whether crime was earning its keep. Which district of the city the property sits in, from
official city boundary data. What citywide prices were doing in the run-up, and what that
particular district was fetching per square metre a month earlier. What the country's central-bank
interest rate was on the day the sale completed, because that is what the buyer's mortgage cost.
And — the piece that turned out to matter more than any of the others — what the property itself
sold for the last time it changed hands, which the file records for 61 % of these sales.

With all of that in front of it, the model estimates what a property was worth. Then §15 wraps
that estimate in a **floor**: a price the property is unlikely to be worth less than, holding
about 90 % of the time, checked against held-out data rather than assumed. A sale is flagged as a
**flip candidate** only when its price falls below its own floor. That is the difference between
"this looks cheap" and a claim with a margin of safety attached.

One rule runs through all of it: **only use what a buyer standing at the transaction date could
have known.** Nearly every design decision below follows from that one line.

*Not from the UK?* Boroughs, LSOAs, leasehold, stamp duty and the various London rail networks are
all defined in the next cell — read it once and the rest of the notebook should follow.


### How the project is laid out

This notebook is the narrative. The pipeline it drives lives in `src/lff/` — one module per
stage, each a small pure function that takes data in and returns data out, so any cell can be
re-run without corrupting another cell's state and the whole run is deterministic given
`CONFIG.seed`. `tests/` holds the leakage probes, which run in seconds against a committed
500-row fixture instead of only at the end of a full run.

| | |
|---|---|
| **Notebook** | the argument and the evidence |
| **`src/lff/`** | the machinery |
| **`tests/`** | the checks |

### What the finished pipeline found

Full run over 59,946 in-window transactions, scored on a held-out test split that nothing before
§14 reads:

| Result | Figure |
|---|---|
| Selected model (chosen on validation MdAPE) | `CatBoost detrended-market (cleaned)` — **13.88 %** test MdAPE |
| Ridge baseline | 16.62 % test |
| Plain gradient-boosted trees | 18–20 % test — **worse than the linear baseline**, until §12.1 fixes the target |
| Strongest feature group | a property's **own prior sale**: +0.93 pp of validation MdAPE |
| Weakest feature group | **crime** at +0.11 pp — below the 0.15 pp bar it had to clear to justify the 2008–2016 window |
| Safety bound | a multiplicative conformal floor at a 90 % target, with empirical coverage reported in §15 |

The headline result is not the model architecture. It is that **the target had to be reframed**:
predicting a property's price *relative to the market level* rather than its price outright is
what lets gradient-boosted trees beat a linear baseline at all (§12.1).

## Reading this without London knowledge

The data is British, and so is a lot of the vocabulary below. None of it is decoration —
`tenure`, `borough` and `outcode` are model features, stamp duty is a line item in the
profit calculation §18 proposes, and the difference between London's rail networks is the whole
reason §6 filters the station file. Everything you need is here; nothing is explained twice.

| Term | What it means |
|---|---|
| **Borough** | One of London's 33 local-government districts — the coarsest geography used here. Each holds roughly 150,000–400,000 people. |
| **LSOA** | *Lower Layer Super Output Area*, the UK census's small-area unit: about 1,500 residents each. 4,835 of them cover London, so roughly 147 per borough. When §8.2 argues about "resolution", this is the fine end of it. |
| **Postcode**, **outcode** | A UK postal code looks like `SW1A 1AA`. The **outcode** is the first half (`SW1A`) and covers a few thousand addresses — finer than a borough, coarser than a street. |
| **Freehold** vs **leasehold** (`tenure`) | Freehold means you own the building and the land outright. Leasehold means you own the right to occupy it for a fixed number of years — often 99 to 999 at the start — and the property loses value as that term runs down. Most London flats are leasehold, which is why "a short lease" turns up later as a reason a property is genuinely cheap rather than mispriced. |
| **Flat** | An apartment. |
| **Terraced**, **semi-detached**, **detached** | A row house joined on both sides, a house joined on one side, and a free-standing house. |
| **The Underground** ("the tube"), **Overground**, **DLR**, **Elizabeth Line**, **Tramlink** | London's rail networks: the subway; the suburban rail network; an automated light-rail line serving the redeveloped eastern docklands; a fast east–west line that opened in 2022; and street-running trams in one southern suburb. **TfL** (Transport for London) runs all of them. |
| **Fare zone** | London's transport pricing is arranged in concentric rings, Zone 1 at the centre out to Zone 6 at the edge — which makes the zone number a decent shorthand for how central an address is. |
| **Land Registry** | The government body that records every property sale in England and Wales. The price history used here derives from it. |
| **Bank of England base rate** | The UK central bank's policy interest rate — what mortgage rates are priced off, and therefore what a buyer's monthly cost depends on. |
| **Met Police** | The Metropolitan Police, the force covering Greater London. The crime data is theirs. |
| **Stamp duty** | A tax the **buyer** pays when purchasing a property, charged in bands, with an extra 3 % since 2016 on any property that is not your main home — a real and substantial cost for anyone flipping. |
| **Units** | Prices are in pounds sterling (£). Floor areas are in square metres; 1 sqm ≈ 10.8 sq ft. |

## How to read this notebook

The notebook runs top to bottom with no manual setup — §3 downloads the data if it is missing.
Sections are grouped into seven phases; each phase answers one question. If any of the British
property or transport vocabulary is unfamiliar, everything is defined once in
[Reading this without London knowledge](#reading-this-without-london-knowledge) above.

| Phase | Sections | The question it answers |
|---|---|---|
| **Setup** | [1](#1-imports-and-environment) · [2](#2-configuration) · [3](#3-data-acquisition) | What are we working with, and where does it come from? |
| **Data** | [4](#4-loading-the-raw-sources) · [5](#5-cleaning-and-per-source-feature-construction) · [6](#6-spatial-engineering) · [7](#7-building-the-master-table) | How do four raw sources become one modelling table? |
| **Exploration** | [8](#8-exploratory-data-analysis) | What actually moves price in this market? |
| **Features** | [9](#9-temporal-and-market-feature-engineering) · [10](#10-chronological-partitioning-and-encoding) | What can the model see without seeing the future? |
| **Models** | [11](#11-loss-functions-metrics-and-decision-rules) · [12](#12-models) · [13](#13-validation-leaderboard) | What do we train, against what loss, and which wins? |
| **Decision** | [14](#14-held-out-test-evaluation) · [15](#15-conformal-safety-bound-and-the-flip-scanner) | How good is it really, and what can a buyer act on? |
| **Assurance** | [16](#16-persisting-the-run) · [17](#17-automated-self-checks) · [18](#18-limitations-and-where-to-take-this-next) | What is persisted, what is asserted, what is still wrong? |

<details>
<summary><b>Full section index</b></summary>

- [Reading this without London knowledge](#reading-this-without-london-knowledge)
- [1. Imports and environment](#1-imports-and-environment)
- [2. Configuration](#2-configuration)
- [3. Data acquisition](#3-data-acquisition)
- [4. Loading the raw sources](#4-loading-the-raw-sources)
- [5. Cleaning and per-source feature construction](#5-cleaning-and-per-source-feature-construction)
- [6. Spatial engineering](#6-spatial-engineering)
- [7. Building the master table](#7-building-the-master-table)
- [8. Exploratory data analysis](#8-exploratory-data-analysis)
  - [8.1 Spatial, safety and macroeconomic drivers](#81-spatial-safety-and-macroeconomic-drivers)
  - [8.2 Does crime price into London property?](#82-does-crime-price-into-london-property)
  - [8.3 The repeat-sales structure of the file](#83-the-repeat-sales-structure-of-the-file)
- [9. Temporal and market feature engineering](#9-temporal-and-market-feature-engineering)
- [10. Chronological partitioning and encoding](#10-chronological-partitioning-and-encoding)
- [11. Loss functions, metrics and decision rules](#11-loss-functions-metrics-and-decision-rules)
- [12. Models](#12-models)
  - [12.1 Detrending the target: a fix, not just a diagnosis](#121-detrending-the-target-a-fix-not-just-a-diagnosis)
- [13. Validation leaderboard](#13-validation-leaderboard)
- [14. Held-out test evaluation](#14-held-out-test-evaluation)
  - [14.1 Repeat-property diagnostic](#141-repeat-property-diagnostic)
  - [14.3 Choosing a target transform: three options, tested head to head](#143-choosing-a-target-transform-three-options-tested-head-to-head)
  - [14.5 Feature-group ablation: what is each block of features actually worth?](#145-feature-group-ablation-what-is-each-block-of-features-actually-worth)
  - [14.6 What a property's own history is worth](#146-what-a-propertys-own-history-is-worth)
- [15. Conformal safety bound and the flip scanner](#15-conformal-safety-bound-and-the-flip-scanner)
- [16. Persisting the run](#16-persisting-the-run)
- [17. Automated self-checks](#17-automated-self-checks)
- [18. Limitations and where to take this next](#18-limitations-and-where-to-take-this-next)
- [Appendix: checking against real, current listings](#appendix-checking-against-real-current-listings)
- [Appendix: related work](#appendix-related-work)

</details>

---
## 1. Imports and environment

**What happens here.** The pipeline is imported from `src/lff/`. Nothing below this cell
re-imports anything, and nothing installs packages at runtime — dependencies come from
`requirements.txt` (see `README.md`).

| Module | Stage |
|---|---|
| `lff.config` | paths, tunables, seeding (§2) |
| `lff.ingest` | dataset download and raw reads (§3–4) |
| `lff.clean` | per-source cleaning (§5) |
| `lff.spatial` | projection, nearest-neighbour and point-in-polygon joins (§6) |
| `lff.master` | the joined master table (§7) |
| `lff.features` | temporal, market, prior-sale, and the feature registry (§9) |
| `lff.split` | chronological partitioning, encoding, training variants (§10) |
| `lff.metrics` | one metric implementation, one results registry (§11) |
| `lff.models` | trainers, deflators, mixture-of-experts (§12) |
| `lff.analysis` | diagnostics and design studies (§12.1, §14.1, §14.3, §14.5–14.6) |
| `lff.conformal` | conformal bound and flip scanner (§15) |
| `lff.plots`, `lff.maps` | every figure |
| `lff.persist` | run artifacts and self-checks (§16–17) |

Warnings are filtered *narrowly* rather than with a blanket `filterwarnings('ignore')`, so genuine
problems still surface. `apply_notebook_theme()` installs those two filters along with the chart
theme and frame display width.

In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

# Importable from a clone with no install step.
_SRC = Path.cwd() / "src"
if _SRC.exists() and str(_SRC) not in sys.path:
    sys.path.insert(0, str(_SRC))

import geopandas as gpd
import numpy as np
import pandas as pd
import xgboost as xgb

import lff
from lff.analysis import (
    ablation_study,
    crime_resolution_study,
    extrapolation_bias,
    prior_sale_study,
    repeat_property_diagnostic,
    summarise_target_transform,
)
from lff.clean import build_crime_features, build_rate_curve, clean_houses
from lff.config import Config, set_seeds
from lff.conformal import calibrate_conformal, scan_for_flips
from lff.crime import build_crime_features_lsoa
from lff.external import fetch_lsoa_boundaries
from lff.features.market import add_market_features
from lff.features.prior_sale import (
    add_prior_sale_features,
    assert_no_lookahead,
    build_sale_history,
)
from lff.features.registry import (
    CATEGORICAL_FEATURES,
    FEATURE_GROUPS,
    FEATURES,
    NUMERIC_FEATURES,
    TARGET,
)
from lff.features.temporal import add_temporal_features
from lff.ingest import ensure_dataset, load_raw
from lff.maps import (
    plot_crime_and_price_maps,
    plot_crime_change,
    plot_crime_within_borough,
)
from lff.master import build_master_table
from lff.metrics import ResultsRegistry, evaluate
from lff.models import train_all
from lff.notebook import apply_notebook_theme
from lff.persist import persist_run, run_self_checks
from lff.plots import (
    plot_ablation,
    plot_crime_and_market,
    plot_flip_margins,
    plot_leaderboard,
    plot_price_clip_comparison,
    plot_price_per_sqm_by_borough,
    plot_price_vs_area,
    plot_property_characteristics,
    plot_tube_premium,
)
from lff.spatial import split_station_networks
from lff.split import chronological_split, training_variants

apply_notebook_theme()

import catboost  # noqa: E402  -- imported here purely to report its version
import sklearn  # noqa: E402

print(f"python      {sys.version.split()[0]}")
for _mod in (np, pd, gpd, xgb):
    print(f"{_mod.__name__:<12}{_mod.__version__}")
print(f"{'sklearn':<12}{sklearn.__version__}")
print(f"{'catboost':<12}{catboost.__version__}")
print(f"{'lff':<12}{lff.__version__}  (src/lff -- see README for the module map)")

---
## 2. Configuration

**What happens here.** One frozen `Config` object becomes the only source of truth for paths,
filters, split ratios and hyper-parameters. Nothing below hardcodes a threshold — change a value
here and re-run.

- **`FAST_MODE`** shrinks every model's iteration budget so the notebook executes in a couple of
  minutes as a smoke test. Set `LFF_FAST_MODE=1` in the environment to enable it.
- **Data location** resolves in this order: `$LFF_DATA_DIR` → `./data/data_for_ds_project`.
  §3 downloads the data if it is not there.
- **`ABLATION_GATE_PP`** is defined here rather than at its point of use because two separate
  studies (§14.5 and §14.6) must apply the *same* bar. It is the decision rule for whether a
  feature group earns the data it costs — see §11.

In [ ]:
CONFIG = Config()
CONFIG.artifact_dir.mkdir(parents=True, exist_ok=True)

set_seeds(CONFIG.seed)

print(f"data_dir     {CONFIG.data_dir}")
print(f"artifact_dir {CONFIG.artifact_dir}")
print(f"fast_mode    {CONFIG.fast_mode}")
print(f"split        {CONFIG.train_frac:.0%} train / {CONFIG.val_frac:.0%} val / "
      f"{CONFIG.calib_frac:.0%} calib / {CONFIG.test_frac:.0%} test")

# The bar a feature group must clear to justify the data it costs. Defined here rather than in
# section 14.5 because the prior-sale study in 14.6 consumes it too, and both must use the same
# number -- see section 11 for the full set of decision rules.
ABLATION_GATE_PP = 0.15

---
## 3. Data acquisition

**What happens here.** The datasets total ~1.8 GB, far past GitHub's file limit, so they ship as
a release asset rather than living in the repository. This cell makes the notebook
self-bootstrapping: if the files are missing it downloads and extracts them; if they are already
present it does nothing.

That is what makes the notebook runnable on a fresh machine with no manual setup.

In [ ]:
ensure_dataset(CONFIG)

---
## 4. Loading the raw sources

**What happens here.** Four sources are read into memory, plus two extra frames that §8.2 needs
at a finer grain.

| Source | Grain | Contributes |
|---|---|---|
| Land-Registry-derived price history | one row per sale event | target price + physical attributes |
| Met Police crime by LSOA | LSOA × category × month | neighbourhood safety |
| Bank of England base rate | one row per rate change | cost of borrowing |
| TfL station geodata (Feb 2022 snapshot) | one row per station | transport connectivity |
| GLA borough boundaries | polygon per borough | administrative and spatial context |

**One deliberate exclusion.** The source file carries `saleEstimate_*` and `rentEstimate_*`
columns — a third party's *model output* for the same property. Using them to predict price would
be target leakage dressed up as a feature, so they are never read.

The crime file is read twice on purpose: once at borough grain for the main pipeline, and once at
its native **LSOA** grain for §8.2, which tests whether the borough aggregation is throwing away
the signal.

In [ ]:
RAW = load_raw(CONFIG)

# Two extra sources for section 8.2. Boundaries come from ONS and are cached under
# data/external/; the crime file is re-read at its native LSOA grain, which load_raw
# deliberately does not do -- it reads only the four columns the borough aggregation needs.
LSOA_BOUNDS = fetch_lsoa_boundaries(CONFIG)
CRIME_LSOA_RAW = pd.read_csv(
    CONFIG.crime_csv,
    usecols=["lsoa_code", "major_category", "value", "year", "month"],
    dtype={"lsoa_code": "category", "major_category": "category",
           "value": "int16", "year": "int16", "month": "int8"},
)

---
## 5. Cleaning and per-source feature construction

**What happens here.** Three independent, pure transformations — each takes one raw frame and
returns a tidy one. No cross-source joining yet; that is §6 and §7.

**Crime is engineered with two lags, never with contemporaneous data.** `crime_volume` is the
month immediately *before* the sale (a short-term safety signal); `crime_volume_prev_12m` is the
rolling twelve-month sum computed with `closed='left'`, so the current month is excluded (a
stable "reputation" signal). Both are strictly backward-looking — what a buyer standing at the
transaction date could actually have known.

**Interest rates** arrive as a sparse list of change dates, forward-filled onto a daily calendar
so every sale is matched to the rate in force on the day it completed.

In [ ]:
HOUSES = clean_houses(RAW["houses"], CONFIG)
CRIME = build_crime_features(RAW["crime"])
RATES = build_rate_curve(RAW["boe"], CONFIG)
LSOA_CRIME = build_crime_features_lsoa(CRIME_LSOA_RAW, LSOA_BOUNDS)

---
## 6. Spatial engineering

**What happens here.** Location is the single biggest driver of price in this market, so the
spatial work gets more care than anything else in the pipeline.

**Projection comes first.** Before any distance is computed, everything is reprojected to the
British National Grid (EPSG:27700). Latitude and longitude are poor units for distance: at
London's latitude a degree of longitude covers only about 62 % of the ground a degree of latitude
does, so measuring in degrees quietly distorts geography along the east–west axis. BNG is metric,
so a distance of `1000` means 1,000 metres in any direction.

**Nearest-station lookup uses a k-d tree (`cKDTree`).** Brute-forcing ~80,000 properties against
~400 stations is 32 million distance calculations for no benefit. A k-d tree answers each query in
`O(log n)`, finishing the join in well under a second — and the same lookup returns the neighbour's
index, which hands over the station's fare zone for free.

### The station file needed cleaning up first

`Stations_20220221.csv` is a February 2022 snapshot of the TfL network, but every transaction here
happened before 2017. Used as-is, it would credit properties with transport links that did not yet
exist:

| Group | Count | Treatment |
|---|---:|---|
| London Underground | 270 | kept — includes the 6 that later *also* gained Elizabeth Line service |
| London Overground | 113 | kept in the wider transit measure |
| DLR | 45 | kept in the wider transit measure |
| Elizabeth-Line-only | 33 | **excluded** — the line did not open until May 2022 |
| Tramlink-only | 39 | **excluded** — street-running trams in one southern suburb, not a rail link that changes how connected a property is |

Filtering on the Underground/Overground/DLR flags handles both exclusions in one pass, and avoids
a subtler mistake. A handful of stations — Paddington, a major central interchange, among them —
had been Underground stops for over a century and merely *gained* Elizabeth Line service on top in
2022. Excluding anything tagged "Elizabeth Line" outright would erase a transport link that
genuinely existed at the time of sale, rather than just dropping the part of the station that did
not exist yet.

**Two distance features, not one**, because they answer different questions:
`distance_to_underground_m` measures proximity to the Underground specifically;
`distance_to_transit_m` measures proximity to any rail-based mode. An earlier undifferentiated
measure computed across all 471 stations treated a suburban tram stop as the equivalent of a
central Underground station — which is not how anyone buying a home would see it.

> **Known limitation.** Several Overground and DLR extensions opened *during* the 2008–2016 window,
> so transit distance is likely overstated for properties sold in the earlier years. A correct
> treatment needs station-level opening dates, which this dataset does not carry. The feature is
> kept anyway: the affected extensions are a small minority of the network, the bias is
> one-directional and partial rather than a fabricated link, and dropping transit connectivity
> entirely would discard a materially stronger signal than the one it would correct for.

In [ ]:
# The station snapshot is filtered here so section 6's narrative has visible output; the
# same split runs inside build_master_table.
UNDERGROUND, HEAVY_RAIL = split_station_networks(RAW["stations"])

---
## 7. Building the master table

**What happens here.** One orchestration function chains §4–§6 into the modelling table. It is
deliberately the only place that writes `df_master`, and it takes no globals other than `CONFIG`,
so re-running it always produces the same result.

Row filters applied here, and why:

* **`price_per_sqm >= 1500`** — removes symbolic transfers that are legally sales but
  economically meaningless: £1 transfers between family members, parking spaces, and lease
  extensions (a leaseholder paying to add years back onto their term, which the register logs as a
  sale). This is a *ratio* filter, so genuinely expensive homes survive as long as their
  price-to-size ratio is plausible.
* **Deduplication** on date + geometry + size + price — the same completion occasionally appears
  more than once in the history file.
* **Chronological sort** — mandatory before any rolling window or time-based split.

The result is cached to Parquet so later runs skip the ~1 GB crime read.

> Note for §18: both this £1,500/sqm floor and the £4 M cap in §10 are defined *using the target*.
> They are defensible as definitions of "the standard market this product serves", but they are
> not identifiable at prediction time — a caveat §18 returns to.

In [ ]:
df_master = build_master_table(CONFIG, HOUSES, CRIME, RATES, RAW["stations"],
                               lsoa_crime=LSOA_CRIME, lsoa_boundaries=LSOA_BOUNDS)
df_master.head(3)

---
## 8. Exploratory data analysis

**What happens here.** Before any modelling, four questions: what does the price distribution look
like, what physical attributes move it, what external forces move it, and what structure does the
file itself have that we might be ignoring?

Each figure is a function of a DataFrame, called once, and nothing here mutates `df_master` or
rebinds a source frame. That discipline is what keeps the notebook re-runnable top to bottom — an
EDA cell that reassigns a name like `df_tube` silently destroys the raw station table for every
cell below it.

**Chart conventions.** Colours come from a validated categorical palette applied in fixed slot
order, so a colour always means the same series. Single-series charts get one flat hue and no
legend (the title names the series); the correlation matrix gets a diverging blue↔red ramp with a
neutral grey midpoint, because its data has a meaningful zero. Grid and axes stay recessive.

**Charts clip at the 95th percentile of price.** Without it, a handful of £20 M sales stretch every
axis, collapsing the £200k–£800k range where most of the market sits into an unreadable sliver.
This is a *display* choice only — the models below see untruncated data. The figure directly below
shows both versions so the effect is visible rather than asserted.

In [ ]:
plot_price_clip_comparison(df_master, CONFIG)

**What this shows.** The same price column twice: the full distribution on the left, clipped to
the 95th percentile on the right.

**Takeaway.** In the full-data panel the bulk of London's sales bunch hard against the left edge,
because a small number of multi-million-pound properties stretch the axis far enough to flatten
everything else into a spike. Clipped, the same data resolves into a proper right-skewed
distribution with a visible peak and shoulder. The shape underneath is identical; only its
legibility changes. Every other chart in this section uses the clipped view for this reason.

In [ ]:
plot_property_characteristics(df_master, CONFIG)

**What this shows.** Four panels: the price distribution, price against total rooms, median price
by property type, and a correlation matrix over the numeric features.

**Takeaways.**

- **Price is heavily right-skewed even after clipping** — a sharp peak around £250k–£300k and a
  long tail beyond it. This is why every model below trains in **log space** (§11): on the raw
  scale, the tail dominates the loss.
- **Price rises with room count**, from a median around £320k at two rooms to roughly £950k at
  eight — but the boxes widen and overlap heavily at the top end. Room count alone does not pin
  down price for larger properties.
- **Detached properties top the median-price ranking** by property type, roughly double the
  cheapest terraced categories.
- **Size and location are the strongest correlates.** `floorAreaSqM` (r = 0.57) and `bathrooms`
  (r = 0.52) lead the positives; `distance_to_center_m` (r = −0.36) and
  `distance_to_underground_m` (r = −0.30) lead the negatives.
- **The features are heavily collinear.** `floorAreaSqM` and `total_rooms` correlate at r = 0.85 —
  several nominally independent drivers are measuring the same thing, size, from different angles.
  Hold on to this: it is the reason §14.5 finds that removing whole feature groups costs almost
  nothing.

### 8.1 Spatial, safety and macroeconomic drivers

**What happens here.** Three external forces — transport, crime, and the cost of borrowing — each
shown on its own scale.

The price-versus-interest-rate figure deliberately uses **two stacked panels sharing one x-axis
rather than one chart with twin y-axes**. A dual-axis chart lets whoever draws it decide where the
two lines appear to cross by choosing the scales, manufacturing a visual correlation that may not
exist. Stacked panels show the same co-movement without that degree of freedom.

In [ ]:
plot_tube_premium(df_master, CONFIG)
plot_crime_and_market(df_master, CONFIG)
plot_price_vs_area(df_master, CONFIG)
plot_price_per_sqm_by_borough(df_master, CONFIG)

**What this shows.** Four charts: mean price by distance to the nearest station, price by
neighbourhood crime band, price against floor area on log-log axes, and median price per square
metre by borough.

**Takeaways.**

- **The tube premium is steep.** Mean price falls from £900k–£950k within 500 m of a station to
  under £350k beyond 3 km. The very first band (0–250 m) is not the cheapest — likely a shorter
  walk traded against living right next to the station itself.
- **Price tracks floor area closely** on a log-log scale, with the spread widening at the top end
  where finish and location start to matter as much as square metreage.
- **Borough spans roughly 5×** in median price per square metre — from over £11,000 in Kensington
  and Chelsea, the wealthiest borough and one of the most central, down to around £2,300 in Bexley
  on the outer south-eastern edge. That ratio is the centre-to-edge gradient of the city in a
  single number, and it is why `borough` carries so much of the location signal.
- **The crime panel is backwards, and that is the interesting part.** Median price *climbs* from
  the `Low` band through to `Severe`, topping out over £470k against roughly £390k in `Low`. Read
  naively that says more crime means higher prices. §8.2 unpicks it.
- **Cheap borrowing and rising prices move together.** The base rate collapses from 5.5 % to 0.5 %
  across 2008–2009 while price dips and then climbs steadily for years afterward.

> ⚠️ **Read these as *marginal* associations, not as evidence the model relies on them.** Every
> chart above answers "does price vary with this?" — a different question from "does this add
> anything the other features do not already carry?". §14.5 asks the second question by removing
> each group from the fitted model, and for **transport** and **interest rate** the answers
> diverge sharply. The reconciliation is tabulated after the ablation in §14.5.

### 8.2 Does crime price into London property?

**The question.** The boxplot above says higher-crime neighbourhoods carry *higher* prices —
backwards from what buyers report caring about. Either crime genuinely does not price into this
market, or something else is driving both variables at once.

This matters far beyond one feature. **The crime file only covers 2008–2016, and that single
constraint is why the entire modelling window stops there** — 59,946 in-window sales out of
314,895 distinct sales in the full history, so 81 % of the available record is discarded to
accommodate one dataset. A feature that expensive has to earn its place.

Before that verdict can be taken, three things about how crime is measured have to be fixed,
because each one stacks against it:

* **Resolution.** The raw file (`london_crime_by_lsoa.csv`) carries 4,835 LSOA codes.
  `build_crime_features` sums them to 33 boroughs — a **147× loss**. To put that in concrete
  terms, one borough mean averages Hampstead, among the most expensive neighbourhoods in the
  country, together with Kilburn, a far cheaper area barely a mile down the road. Any relationship
  between local safety and local price is overwhelmingly a *within*-borough effect — exactly the
  variation a borough mean erases.
* **Count, not rate.** `crime_volume` is a raw count, so a borough with more people in it scores
  high automatically. Because every LSOA holds roughly the same 1,500 residents by construction,
  an LSOA count is already close to a per-capita rate without dividing by anything.
* **One series.** Burglary and drug offences are summed together, assuming a buyer prices them
  identically.

All three are fixed here, at LSOA grain, before any conclusion is drawn. The point is not to
rescue crime. It is that *"crime does not matter"* and *"crime measured 147× too coarsely, as an
unnormalised count, does not matter"* are different claims, and only the second has been tested.

In [ ]:
# Left: what a place costs. Right: how much crime it records. One hue each, so the two
# panels cannot be misread as sharing a scale.
lsoa_gdf = plot_crime_and_price_maps(df_master, LSOA_CRIME, LSOA_BOUNDS)

**What this shows.** Two choropleths of Greater London at LSOA grain — price on the left, recorded
crime on the right. One hue each, so the panels cannot be misread as sharing a scale.

**Takeaway: the confound is visible.** Both maps run darkest in the middle. Central London is
simultaneously the most expensive *and* the most crime-recording part of the city, so the raw
association comes out **positive** — naively, more crime looks like more money. The crime-quartile
boxplot in §8.1 has exactly this confound baked into it. (2,562 of the 2,863 Greater London LSOAs
record at least five sales in the window and are used here.)

**Two ways to strip centrality out**, each shown below:

1. **Within-borough.** Subtract each borough's own mean from both variables, leaving the contrast
   between two streets in the same borough — one safer than the other.
2. **Differenced over time.** Remove every fixed feature of a place at once — architecture, parks,
   distance to the centre, reputation — and ask a sharper question: did the LSOAs where crime
   *fell* see faster price growth than the ones where it rose?

In [ ]:
crime_partial = plot_crime_within_borough(lsoa_gdf)

In [ ]:
crime_differenced = plot_crime_change(df_master, LSOA_CRIME)

**The association survives neither treatment.**

- **Within-borough:** the correlation drops from r = +0.30 across boroughs to **r = +0.00** within
  them. The entire apparent relationship is *which borough a property sits in*, not its crime
  level.
- **Differenced over time:** **r = −0.01** between the change in local crime and the change in
  local price, across 863 LSOAs — the same verdict from an unrelated direction.

Whatever the raw scatter measures, it is not crime.

**What this does and does not settle.** This is the exploratory answer. A feature can carry no
marginal correlation and still earn its place by interacting with others inside a model, so §14.5
puts the same question to the fitted model — same gate, five seeds, six crime designs — and reports
what it finds there.

### 8.3 The repeat-sales structure of the file

**What happens here.** One structural fact about the dataset that none of the charts above
surface, and that turns out to matter more than any of them.

**The source file is a price *history*, not a transaction list.** 418,201 rows cover 137,760
unique addresses. Once the 24.5 % of rows that are exact duplicates are removed (315,674
remain) and the 779 same-day price conflicts are collapsed to a median, **314,895 distinct
sales** remain — and **66.0 % of addresses still record two or more of them**. Within the
modelling window, **61.1 % of sales have an earlier sale of the same property somewhere in the
file**, typically about seven years earlier.

**Why this is the most important cell in §8.** Every other feature in this project asks *what is a
property like this worth?* A prior sale asks a different and much easier question: *what was this
exact property worth last time, and how far has the market moved since?* The property's own
quality, layout, aspect, street and lease are all held fixed between the two observations — which
is precisely the logic behind the repeat-sales index literature (Bailey, Muth & Nourse 1963; see
the [appendix](#appendix-related-work)).

The features are built in §9, and §14.6 measures what they are worth — on the same gate and the
same paired-seed protocol as every other feature group, so the verdict is comparable to crime's
rather than argued for separately.

---
## 9. Temporal and market feature engineering

**What happens here.** Three families of feature get built — temporal, market-level, and
prior-sale. This is where most of the leakage risk in a property model lives, so every feature is
built to answer one question: *would a buyer standing at the transaction date have known this?*

### Temporal and market features

| Feature | Construction | Why it cannot leak |
|---|---|---|
| `days_since_start` | days elapsed since the first transaction | derived from the row's own date |
| `month_sin`, `month_cos` | cyclic encoding of calendar month | December and January end up adjacent, as they are in reality |
| `market_median_rolling_3m` / `_12m` | market-wide monthly median, `.shift(1)` then rolled | the shift drops the current month before the window opens |
| `lagged_borough_median_sqm` | borough £/sqm median, stamped onto the *following* month | a month's own price never informs its own prediction |
| `avg_room_size` | floor area ÷ total rooms | a within-row ratio |

`avg_room_size` needs care: `total_rooms` can be zero, which yields ±∞ rather than NaN. Those are
converted to NaN explicitly and then **left as NaN**. Filling them with the column median here
would leak — this function runs on the whole table, before the split, so that median would be
computed over validation, calibration and test rows and then baked into a training feature. Small,
but real. Leaving NaN defers imputation to each model, where it happens inside a pipeline fitted on
the training split alone.

### Prior-sale features (§8.3)

| Feature | Construction | Why it cannot leak |
|---|---|---|
| `prev_sale_price` | price of the most recent sale of the *same address*, strictly before this row's date | an as-of merge with a strict `<` bound |
| `years_since_prev_sale` | elapsed time between the two sales | both dates precede the prediction |
| `prev_sale_days_since_start` | *when* that earlier sale happened, on the same clock as `days_since_start` | a past date |
| `has_prev_sale` | whether a match exists at all | presence, not price |

**History is read from the whole 1995–2024 file, not the modelling window.** A 2009 sale's previous
sale is usually pre-2008; clipping first would discard most of the signal.

**"Strictly before" is load-bearing, and three things could break it.** The 102,527 duplicate rows
are removed first, or an as-of merge can return the very row it is meant to predict. Same-day
conflicting prices are collapsed to a median. And `assert_no_lookahead` runs as a hard check
below, with a matching assertion in the §17 self-checks.

Two further columns the module can build — `log_prev_sale_price` and `n_prior_sales` — are
deliberately **not** carried. §14.6 shows they earn nothing.

In [ ]:
# A property's own transaction history, read from the WHOLE 1995-2024 file rather than the
# modelling window: a 2009 sale's previous sale is usually pre-2008, and clipping first would
# discard most of the signal. Section 14.6 measures what it is worth.
SALE_HISTORY = build_sale_history(RAW["houses"])

df_model = add_market_features(add_temporal_features(df_master))
df_model = add_prior_sale_features(df_model, SALE_HISTORY)
df_model = df_model.dropna(subset=[TARGET, "date"]).reset_index(drop=True)
assert_no_lookahead(df_model)

print(f"Modelling table: {df_model.shape}")
print(f"Features: {len(NUMERIC_FEATURES)} numeric + {len(CATEGORICAL_FEATURES)} categorical")
missing = df_model[FEATURES].isna().mean().sort_values(ascending=False)
print("\nMissing-value rate by feature (top 6):")
print(missing.head(6).to_string())

**What this shows.** The sale-history reconciliation, prior-sale match rate, the final modelling
table's shape, and the missing-value rate of the six most incomplete features.

**Takeaways.**

- **The modelling table is 59,946 rows × 28 features** (23 numeric + 5 categorical).
- **61.1 % of rows carry a prior sale**, a median of 6.9 years earlier (p10 2.2, p90 13.8).
- **`prev_sale_*` is 38.9 % missing by construction** — the other 38.9 % of properties simply have
  no earlier sale in the file. This is *informative* missingness, not a data-quality defect, which
  is why `has_prev_sale` is carried alongside: it lets the model treat "never sold before" as its
  own case rather than as a hole to impute. Gradient-boosted trees handle NaN natively; Ridge
  imputes inside a pipeline fitted on the training split alone.
- **`assert_no_lookahead` passed** — no row reads a sale dated at or after its own.

---
## 10. Chronological partitioning and encoding

**What happens here.** The table is cut into four time-ordered slices, then encoded — both fitted
on training rows only.

**The split is by time, not at random.** A random split would let the model learn from June 2016
to predict January 2016, a situation that never occurs in production. Sorting by date and cutting
**60 / 15 / 10 / 15** into train / validation / calibration / test reproduces the real task: train
on the past, forecast the future.

**Why calibration gets its own slice.** One validation set cannot serve three roles at once —
early stopping, model selection, *and* calibrating the conformal bound — without the "90 %
guarantee" in §15 being partly calibrated on data the model was already tuned against. That is
circular, so calibration gets a dedicated slice nothing else touches. The ordering of the two
middle slices was tested both ways; validation-then-calibration is kept because it produces smaller
validation-to-test drift for the tree models, at the cost of a little exchangeability on the
calibration split. (§18 records that this comparison was itself made by scoring on test.)

### One evaluation universe, three training variants

Comparing a "raw", a "capped" and a "cleaned" model is only meaningful if all three are scored on
the same rows; letting the evaluation set vary alongside the training set compares models and test
sets at once. Here the evaluation universe is **fixed** — validation, calibration and test rows
under the £4 M cap, the standard market the product targets — and only the *training* data varies:

| Variant | Training rows |
|---|---|
| `raw` | every transaction, including £4 M+ |
| `capped` | transactions at or below the cap |
| `cleaned` | capped, minus multivariate anomalies |

**Anomalies are removed from training only.** Fitting `IsolationForest` on the whole dataset and
dropping flagged rows everywhere would delete the hard cases from validation and test as well —
marking your own exam after removing the difficult questions. A production model does not get to
refuse the awkward listings, so the forest is fitted on the training slice and filters only that.

### Encoding

Two representations, both fitted on training data only:

* **Native categoricals** (`category` dtype) for XGBoost — `propertyType`, `tenure`, `borough`
  and `outcode`, that last one being the first half of the postcode, so a mid-grained location
  label sitting between borough and street. The level set is pinned from the training split so a
  category means the same integer code everywhere. CatBoost gets raw strings via `cat_features`. These columns must never be coerced with
  `pd.to_numeric(..., errors='coerce')`, which turns all four **entirely NaN** and leaves the
  CatBoost models and every MoE router training on dead columns.
* **Smoothed target encoding** for models that require numeric input (the routers). The smoothing
  pulls low-frequency categories toward the global mean, so a borough with three sales does not
  get a confident price estimate.

In [ ]:
SPLITS = chronological_split(df_model, CONFIG)

# One fixed evaluation universe for every model: the standard market, under the cap.
VAL_EVAL = SPLITS.val[SPLITS.val[TARGET] <= CONFIG.price_cap]
CALIB_EVAL = SPLITS.calib[SPLITS.calib[TARGET] <= CONFIG.price_cap]
TEST_EVAL = SPLITS.test[SPLITS.test[TARGET] <= CONFIG.price_cap]
print(f"\nEvaluation universe (price <= \N{POUND SIGN}{CONFIG.price_cap:,.0f}): "
      f"{len(VAL_EVAL):,} validation, {len(CALIB_EVAL):,} calibration, "
      f"{len(TEST_EVAL):,} test rows")

**What this shows.** The row count of each slice, and of the fixed evaluation universe within it.

**Takeaway.** Every model from here on is scored on *these exact rows* — differences in the
leaderboard reflect the model and its training data, never a shifting denominator. The calibration
slice sits strictly between validation and test in time and is untouched by anything except §15.

In [ ]:
VARIANTS = training_variants(SPLITS, CONFIG)

---
## 11. Loss functions, metrics and decision rules

**What happens here.** Nothing is trained yet. This section fixes the three things that must be
settled *before* any model runs, because deciding them afterwards is how a project talks itself
into a result: what each model **optimises**, what every model is **scored** on, and what rule
**decides** the winner.

### What the models optimise

All targets are modelled in **log space** (`log1p` in, `expm1` out). Property prices are strongly
right-skewed (§8), and a squared-pound loss on the raw scale is dominated by the expensive tail —
one £3 M house outweighs thirty £100k flats. In log space the loss becomes approximately
*proportional*: a 10 % miss costs the same at £200k as at £2 M, which is what a buyer actually
experiences.

| Model | Trained on | Objective | Why this one |
|---|---|---|---|
| Ridge | `log1p(price)` | squared error, `alpha=1.0` | closed-form, hard to overfit — a baseline should not need tuning to be fair |
| XGBoost (plain) | `log1p(price)` | `reg:squarederror` (library default) | the untuned starting point, kept so §12.1's fix has something to improve on |
| CatBoost (plain) | `log1p(price)` | `MAE` | already median-aligned |
| XGBoost detrended | `log(price / market level)` | `reg:absoluteerror` | matches the reported metric — see below |
| CatBoost detrended | `log(price / market level)` | `MAE` | unchanged from plain, which **isolates** the deflator's effect from the objective's |

**Why absolute error rather than squared error.** Squared error in log space fits the conditional
*mean* of log price — approximately the geometric mean. The headline metric is the *median* of
relative errors. Those are different estimands, so a squared-error model is optimising one thing
and being judged by another. `reg:absoluteerror` on log-ratio space optimises something close to
the metric actually reported. CatBoost already trained with MAE, so leaving it alone means the
XGBoost/CatBoost comparison isolates which of the two changes — the objective or the deflator — is
doing the work.

### What every model is scored on

One `regression_metrics()` implementation, used everywhere. Near-identical variants defined across
several cells are how a leaderboard silently mixes two spellings of one metric, and with them two
different numbers.

| Metric | Reads |
|---|---|
| **`MdAPE`** | **the headline** — median absolute percentage error |
| `MAPE` | mean absolute percentage error, tail-sensitive |
| `MAE`, `RMSE` | pounds — easy to explain, dominated by expensive properties |
| `R2` | variance explained; reported, never decided on |
| `within_25pct` | share of valuations within 25 % — the practical "close enough" rate |

**Why MdAPE is the headline.** MAE is reported in pounds and is intuitive, but it is dominated by
the expensive tail: a 10 % miss on a £3 M house contributes thirty times more than a 10 % miss on a
£100k flat, even though both are equally wrong in the only sense the buyer cares about. The median
of percentage errors is robust to that tail and answers the practical question — *what does a
typical valuation get wrong by?*

### What decides

Every verdict in this notebook resolves against one of these rules. They are fixed here so that no
result below gets to choose its own bar:

| Rule | Value | Applied in | Decides |
|---|---|---|---|
| Model selection | lowest **validation** MdAPE | §13, §14 | which model becomes `BEST` |
| Feature-group gate | `ABLATION_GATE_PP` = **0.15 pp** | §14.5, §14.6 | whether a feature group earns the data it costs |
| Seed-noise floor | gain > **2 × mean sd** across seeds | §14.5, §14.6 | whether an effect is resolvable at all, or is run-to-run noise |
| Coverage target | **90 %** (`1 - conformal_alpha`) | §15, §17 | whether the safety bound holds empirically |

> **The standing protocol: decisions read validation; test is read only to report.** Early
> stopping, model selection, the deflator choice (§14.3) and both feature studies (§14.5, §14.6)
> are all scored on validation. §18 records the two points in this project's history where that
> protocol was not followed.

In [ ]:
RESULTS = ResultsRegistry()

---
## 12. Models

**What happens here.** Fourteen models are trained, but they are not fourteen guesses — they are
a ladder, where each rung exists because the rung below it failed in a specific, diagnosable way.

**1 — Ridge, the linear baseline.** Fast, closed-form, hard to overfit. It sets the floor and
doubles as a sanity check on everything above it: if a gradient-boosted ensemble cannot beat a
straight-line model on tabular data of this shape, the problem is the setup, not the architecture.

**2 — XGBoost and CatBoost on `log(price)`.** The standard answer for tabular hedonic data. Price
responds non-linearly and interactively to its inputs — the value of a second bathroom depends on
borough, the value of proximity to a station depends on the line — and a linear model cannot
express that. CatBoost specifically for its native categorical handling, which avoids hand-encoding the four
non-numeric columns — property type, freehold-or-leasehold, borough, and postcode district.

**3 — The failure.** The boosters **lose to Ridge**, by five points of MdAPE or more. That is not
a verdict on gradient boosting; it is a symptom, and §12.1 diagnoses the mechanism from validation
data alone.

**4 — The detrended target.** Predict a property's price *relative to the market level* rather
than its price outright, so the trees interpolate within a range they have seen instead of
extrapolating past it. Two deflator candidates enter the pool — market-wide and borough-scaled —
and §14.3 decides between them on validation rather than assuming.

**5 — Mixture of Experts: luxury routing.** Splits standard from luxury stock and soft-weights
one expert each. The question it exists to answer: does segmenting this market beat modelling it
whole?

**6 — The 3-seed average, an ensembling baseline.** Fitting the same recipe three times, differing
only by random seed, and averaging the predictions costs nothing conceptually. It is the
plain-ensembling comparison point the luxury MoE has to beat — without it, a "Mixture of Experts"
that merely beats a single model has proved only that ensembling works.

**Both the luxury MoE and the 3-seed average are trained on the detrended recipe as well as the
plain one.** Judging either only on plain `log(price)` — the target that costs the single trees
their accuracy — would test the design against a handicapped baseline and credit it for a fix it
did not make.

### Why no neural network

Deliberate, not an oversight. This is ~60k rows of tabular data with no free text, no images and
no sequence structure; gradient-boosted trees are the better-suited family at this scale, need far
less tuning to be competitive, and handle the heterogeneous categorical/continuous mix natively.
A neural approach would earn its place if the project gained listing descriptions or photographs,
where learned representations beat hand-built features. It has nothing to work with here.

### One uniform interface

Every trainer returns a `ModelBundle` exposing the same `predict(X)` signature. That uniformity is
not cosmetic — it is what lets the leaderboard, the test evaluation and the conformal calibration
all run through one code path. It also closes off a whole class of error: a conformal step that
reaches for a concrete estimator such as `expert_0.predict(X_test)` computes its guarantee from
whatever that name happens to point at, which need not be the model under evaluation, and fails
silently when it is not.

### 12.1 Detrending the target: a fix, not just a diagnosis

**The symptom.** Gradient-boosted trees are normally the stronger choice on tabular data of this
shape, so plain Ridge beating them by five points of MdAPE reads as a symptom, not a verdict on
the architecture.

**The mechanism.** The plain XGBoost/CatBoost models train on `log1p(price)`. A tree's prediction
is a **constant per leaf**, so once a value of a trending feature (`days_since_start`,
`market_median_rolling_3m`) exceeds anything seen in training, every such row lands in the same
boundary leaf and the model flat-lines at the last price level it learned. It cannot extrapolate a
rising market. Ridge suffers less, because it multiplies by a coefficient instead of splitting, and
so at least projects the trend forward — though the residual evidence below shows it is not immune
either, only less exposed.

That mechanism makes a testable prediction: the plain trees should under-predict by a large,
*positive* mean residual — a one-directional level error, not symmetric noise. The cells below
measure exactly that, on validation.

**The fix removes the need to extrapolate at all.** Instead of predicting price, predict the
*ratio* of price to the lagged whole-market median (`market_median_rolling_3m`, already a
leakage-safe feature from §9):

$$y = \log\!\left(\frac{\text{price}}{\text{market level}}\right)$$

"How much is this property worth relative to where the market already is" is close to stationary
across time even while the market itself trends — so a tree only has to *interpolate* within the
range of ratios it saw in training. The market level is multiplied back onto the prediction at
inference time.

**A second change rides along:** the training objective moves to `reg:absoluteerror` for XGBoost,
for the reasons set out in §11. CatBoost already trains with MAE, so only the detrending applies
there — which isolates which of the two changes is doing the work.

**Two deflators enter the candidate pool, not one.** `market_median_rolling_3m` is one number per
calendar month shared by every property. A finer alternative is `lagged_borough_median_sqm ×
floorAreaSqM` — size- and location-scaled, personalised per row. Both are trained for both
backends. **§14.3 decides between them on validation; this section does not pre-judge it.**

These are real competitors, not a side experiment: they are registered in `train_all()` like every
other model, so whichever wins on validation becomes `BEST` and flows through test evaluation, the
repeat-property diagnostic and conformal calibration exactly like any other model would.


In [ ]:
BUNDLES = train_all(SPLITS, VARIANTS, CONFIG, VAL_EVAL, RESULTS)

The mechanism is structural — a leaf constant cannot exceed the largest value it was fitted on,
whichever period you point it at — so if the diagnosis above is right, the same one-directional
error should show up on validation just as clearly as anywhere else.


In [ ]:
_bias_models = ["XGBoost (capped)", "XGBoost detrended-market (capped)", "Ridge (baseline)"]
bias_val = extrapolation_bias(_bias_models, VAL_EVAL, "val", SPLITS, BUNDLES)
print("Signed-residual diagnostic, computed on VALIDATION only:\n")
print(bias_val.to_string(index=False, formatters={
    "Mean residual (val)": "\N{POUND SIGN}{:,.0f}".format,
    "Median residual": "\N{POUND SIGN}{:,.0f}".format,
    "Under-predicted %": "{:.1f}%".format}))

_plain = bias_val[bias_val["Model"] == "XGBoost (capped)"]
_fixed = bias_val[bias_val["Model"] == "XGBoost detrended-market (capped)"]
if not _plain.empty and not _fixed.empty:
    plain_bias = float(_plain.iloc[0]["Mean residual (val)"])
    fixed_bias = float(_fixed.iloc[0]["Mean residual (val)"])
    print(f"\nPlain XGBoost under-predicts validation by a mean of \N{POUND SIGN}{plain_bias:,.0f}; "
          f"detrending moves that to \N{POUND SIGN}{fixed_bias:,.0f}.")
    verdict = ("visible on validation alone" if plain_bias > 0 and abs(fixed_bias) < abs(plain_bias)
               else "NOT reproduced on validation -- treat section 12.1's reasoning with caution")
    print(f"The extrapolation signature is {verdict}, so the fix did not require the test set.")

bias_val

**What this shows.** Mean and median signed residual (actual − predicted) on validation for three
models, plus the share of rows under-predicted.

**Takeaways.**

- **The extrapolation signature reproduces on validation.** Plain XGBoost carries a large positive
  mean residual — it systematically under-predicts, exactly as a model that cannot follow a rising
  market must. Detrending moves that bias sharply toward zero.
- **Ridge complicates the story rather than confirming it.** §12.1 frames this as "trees cannot
  extrapolate, Ridge can". The measurement is more nuanced: Ridge *also* under-predicts, because
  the market rose faster than any linear projection of its training window. What separates the
  models is the **size** of the level error, not its presence — plain trees worst, Ridge in
  between, detrended trees least biased.
- **So the accurate claim is:** a non-stationary target biases *every* model here and punishes the
  trees hardest, and **detrending, not architecture, is what removes most of it.** The fix in §12.1
  is aimed at that shared cause, not at gradient boosting specifically.

---
## 13. Validation leaderboard

**What happens here.** Every model trained above is ranked on the validation set. This is the table
that **selects** the winner; §14 then scores that winner on data nothing has read yet.

The table is generated from the results registry — every row was appended by `evaluate()` at
training time, so the leaderboard cannot disagree with what the models actually scored. A
hand-assembled table, with metric dictionaries typed into a `DataFrame` literal, carries no such
guarantee: that is how a £4 M cap ends up labelled "<£5M".

Read `Trained on` and `MdAPE` together: every row is scored on the *same* validation rows, so
differences reflect the model and its training data, nothing else.

In [ ]:
val_board = RESULTS.frame("val")
plot_leaderboard(val_board, "Validation performance, identical evaluation rows")
val_board.style.format({
    "R2": "{:.3f}", "MAE": "\N{POUND SIGN}{:,.0f}", "RMSE": "\N{POUND SIGN}{:,.0f}",
    "MdAPE": "{:.2f}%", "MAPE": "{:.2f}%", "within_25pct": "{:.1f}%",
}).background_gradient(cmap="Blues_r", subset=["MdAPE", "MAE"])

**What this shows.** All trained models ranked by validation MdAPE, with MAE alongside — the bar
chart repeats both so the ordering is visible at a glance.

**Takeaways.**

- **The detrended models sweep the top.** Every model trained on `log(price / market level)`
  outranks every model trained on `log(price)` — the single largest effect in the entire
  leaderboard, and confirmation that §12.1 diagnosed the right problem.
- **The plain boosters badly underperform the detrended ones**, and on the held-out set in §14
  they fall behind even the linear baseline. A Ridge regression beating gradient boosting on
  tabular data of this shape is the anomaly the ladder in §12 was built to explain: it signals a
  problem with the target, not with the architecture.
- **MdAPE and MAE do not always agree on the ordering.** They are answering different questions —
  typical percentage miss versus average pound miss, the latter dominated by expensive properties.
  Selection uses MdAPE (§11); MAE is reported so disagreements are visible rather than hidden.

Whichever model tops this table on MdAPE becomes `BEST`. **Nothing has read the test split yet.**

---
## 14. Held-out test evaluation

**This is the section that matters.**

Every number in §13 comes from the validation set — the same data used for early stopping and for
choosing between architectures. Reporting those figures as the model's accuracy is circular: they
measure how well the winner fits the set it won on. **Nothing before this cell has read the test
split.**

The procedure: pick the winner by **validation** MdAPE, then score on test. The gap between the two
is the honest measure of how much validation performance was selection effect.

**How often test is read, precisely.** Every model is scored once here so the drift column exists
and the leaderboard is comparable; the repeat-property diagnostic (14.1) then reads it for the
selected model, and §15 reads it once more to report conformal coverage. That is *reporting*,
not selection — the winner was fixed by validation before this cell ran, and every **decision**
in 14.3–14.6 is made on validation.

**Why the subsections come in this order:**

| | Asks |
|---|---|
| **14.1** | *How good is the model we just revealed?* — the memorisation check |
| **14.3** | *Did the design choices earn their place?* — which target transform to use |
| **14.5–14.6** | *What should the next data pull look like?* — whether crime justifies the window it forces, and what a property's own history is worth |

The arc is deliberate: questions about what we built, then questions about what to build next. The
last two are not properties of *this* model at all — they are scoping decisions for the next
iteration, which is why they come after the design verdicts rather than before them.

> **Why the numbering skips 14.2 and 14.4.** Both sections were removed: a residual-and-APE
> diagnostic that §14.5's ablation answers more rigorously, and a Mixture-of-Experts
> necessity check that only narrated a selection §13 had already fixed. The surviving
> sections keep their original numbers so that every cross-reference in this notebook and
> in `README.md` still points at the section it was written about.

In [ ]:
BEST_NAME = val_board.iloc[0]["Model"]
BEST = BUNDLES[BEST_NAME]
print(f"Selected on validation MdAPE: {BEST_NAME}\n")

print("Scoring every model once on the held-out test set:")
for bundle in BUNDLES.values():
    evaluate(bundle, TEST_EVAL, "test", RESULTS, SPLITS)

test_board = RESULTS.frame("test")
comparison = (
    val_board[["Model", "MdAPE", "MAE"]]
    .merge(test_board[["Model", "MdAPE", "MAE"]], on="Model", suffixes=(" (val)", " (test)"))
)
comparison["MdAPE drift"] = comparison["MdAPE (test)"] - comparison["MdAPE (val)"]
comparison = comparison.sort_values("MdAPE (test)").reset_index(drop=True)
print()
comparison.style.format({
    "MdAPE (val)": "{:.2f}%", "MdAPE (test)": "{:.2f}%", "MdAPE drift": "{:+.2f}pp",
    "MAE (val)": "\N{POUND SIGN}{:,.0f}", "MAE (test)": "\N{POUND SIGN}{:,.0f}",
})

**What this shows.** Every model scored once on the held-out test set, joined to its validation
score, sorted by test MdAPE. The `MdAPE drift` column is test minus validation.

**Takeaways.**

- **The detrended family holds up out of sample.** Every model trained on the reframed target
  occupies the top of the test table, just as it did on validation. §12.1's fix is the one result
  in this project that survives every way of looking at it.
- **Within that family, the ordering does reshuffle — and the selected model is not the test
  winner.** Validation picked `CatBoost detrended-market (cleaned)` (13.88 % test); the best test
  score belongs to `3-seed average detrended (XGB)`, at 13.62 %. That 0.26 pp gap **is** the selection effect, made visible. It is not corrected,
  because correcting it would mean choosing the deployed model on test.
- **Drift is the price of selection.** Every model that early-stopped on validation scores worse
  on test, which is expected: its validation score is optimistic by construction, and the drift
  column is the size of that optimism made visible. **Ridge is the one exception** at −0.47 pp,
  and for a reason that proves the rule — it is closed-form, never early-stopped, and was never
  selected on, so it has no optimism to give back.
- **Ridge's test R² of −180 is a warning label on that baseline, not on the metric.** Its MdAPE
  is a sane 16.62 %, but a handful of test rows extrapolate far enough in log space that
  `expm1` returns an absurd price, which one squared term is enough to dominate. Read Ridge as
  a *typical-error* floor only; MdAPE is the selection metric precisely because it is immune to
  this (§11).
- **The plain boosters remain worse than Ridge on test**, confirming §12.1's diagnosis was
  structural rather than a validation-set artefact.
- **Near-ties at the top should not be over-read.** Where two models sit within ~0.1 pp on
  validation, that gap does not reliably predict which wins on test.

`BEST` was fixed by validation *before* this cell ran and is not revised in light of it. Revising
it here would make test a selection signal, which is the one thing holding it out was meant to
prevent.

### 14.1 Repeat-property diagnostic

**The question: is the headline number measuring valuation, or memorisation?**

The source file is a price history — one row per sale event — so a dwelling that changed hands
three times between 2008 and 2016 contributes three rows. A chronological split cuts on **time**,
not on **property**, which means a flat sold in 2010 (train) and again in 2016 (test) appears on
both sides of the wall.

That is not automatically cheating: forecasting a known building's next sale price is a real
business task, and the lagged features remain strictly historical. But it does mean the headline
test metric blends two very different problems — re-valuing a property the model has already seen,
and valuing one it never has. The cell below separates them, because **the second number is the one
that generalises to new stock.**

This matters more now than it did before §9: prior-sale features make the model *explicitly* better
on exactly the repeat properties this diagnostic isolates.

In [ ]:
repeat_property_diagnostic(BEST, SPLITS, TEST_EVAL)

**What this shows.** Test MdAPE split two ways: properties whose address also appears in the
training split, and properties the model has never seen at any date.

**Takeaway.** Read the never-seen number as the model's honest generalisation to new stock, and the
seen-before number as its performance on the repeat-sales task. If the gap is large, the headline
metric is partly measuring memorisation — and the fix is a group-aware split keyed on
`fullAddress`, which is improvement #1 in §18. The gap is also the clearest single argument for why
prior-sale features work (§14.6): the model does better where it has history, and 61.1 % of rows
have some.

### 14.3 Choosing a target transform: three options, tested head to head

**The open question from §12.1.** Once a market-wide deflator addresses the extrapolation problem,
does a *more granular* one do better — deflating by borough and property size rather than by the
whole market at once? The intuition says yes: a deflator built from genuinely comparable properties
should carry more signal than one lumping the whole city together.

That is a hypothesis, so it is tested rather than adopted. Three candidates, all already trained
above:

1. **Regular** — predict `log(price)` directly (the untransformed target).
2. **Detrended, market-wide** — predict `log(price / market_median_rolling_3m)`, one deflator shared
   by every property sold in the same calendar month.
3. **Detrended, borough-scaled** — predict `log(price / (lagged_borough_median_sqm ×
   floorAreaSqM))`, a deflator personalised to each property's own size and borough.

Option 3 is what the intuition favours. The comparison below reports what the data does, which is
not what that intuition predicts. **The winner is chosen on validation**; the test column is printed
so the reader can judge whether the choice generalised, not so it can make the choice.

In [ ]:
transform_comparison = summarise_target_transform(RESULTS)
print(transform_comparison.to_string(index=False,
      formatters={"Validation MdAPE": "{:.2f}%".format, "Test MdAPE": "{:.2f}%".format,
                  "Test MAE": "£{:,.0f}".format}))

# The winner is chosen on VALIDATION. Picking it by test MdAPE would be using the held-out set to
# make a design decision -- the exact failure mode the limitations section warns about. The test
# column above is reported so the reader can judge whether the choice generalised.
for backend in transform_comparison["Backend"].unique():
    sub = transform_comparison[transform_comparison["Backend"] == backend].set_index("Target transform")
    winner = sub["Validation MdAPE"].idxmin()
    line = (f"\n{backend}: '{winner}' wins on validation at "
            f"{sub.loc[winner, 'Validation MdAPE']:.2f}% MdAPE")
    if "Test MdAPE" in sub.columns:
        test_winner = sub["Test MdAPE"].idxmin()
        agreement = "and is also best on test" if test_winner == winner else \
                    f"but '{test_winner}' scored best on test"
        line += f" -- {agreement}"
    print(line)

transform_comparison

#### Why market-wide, not borough-scaled

**Both detrended options crush the regular target for both backends** — confirming §12.1's
diagnosis: the target's non-stationarity, not the tree architecture, was the problem.

**Between the two detrended options, the simpler and coarser deflator wins** for both backends, and
by a wide margin for CatBoost — the opposite of what the hypothesis predicted.

**Why.** `lagged_borough_median_sqm` is estimated from far fewer sales per month than the
whole-market median — one borough's transactions in a 3-month window, against the entire city's.
Because the deflator is a **divisor**, a noisier deflator injects that noise directly into every
training label it divides, not merely into one more feature the model could choose to down-weight.
The whole-market median, estimated from thousands of sales, does not have that problem.

**And the coarser deflator costs nothing in borough- or size-specific signal.** `borough`,
`floorAreaSqM` and `lagged_borough_median_sqm` itself remain ordinary input features (§9), so the
tree stays free to learn "this borough commands a premium" or "larger properties are worth
proportionally more" — from *uncorrupted* labels. Folding that information into the deflator adds
nothing the tree could not already reach; it only adds the deflator's estimation noise to the
target.

> **The general lesson.** A detrending deflator should remove only what the model architecture
> genuinely cannot learn on its own — here, the market-wide time trend. Anything the model is
> already capable of learning from a feature should stay a feature, not get folded into the label.

**Chosen going forward: the market-wide deflator.** Simpler (one already-computed column, no
dependency on floor area or borough being non-missing), and it wins on validation for both
backends. Whether validation and test agree is stated by the cell itself rather than asserted here.

`XGBoost detrended-market (capped)` is the strongest *single* model on this recipe — though not
necessarily what §14 selects as `BEST`, which is whichever model wins validation overall, including
the Mixture-of-Experts variants.

### 14.5 Feature-group ablation: what is each block of features actually worth?

**The question, and why it is a scoping decision rather than a model diagnosis.** Each feature
group costs something to obtain and maintain. Crime costs the most by far: the file only covers
2008–2016, and **that single constraint is why the entire modelling window stops there** — 59,946
in-window sales against 314,895 distinct sales in the full history, so **81 % of the record is
discarded to accommodate one feature block.**

The rule adopted at the outset was to start with the smaller window crime supports and widen it
only if crime proved not to matter much. That makes measuring the contribution a prerequisite, not
an afterthought.

**The method.** Retrain **the winning recipe** — XGBoost on the detrended target, `capped` variant,
the configuration §12.1 and §14.3 arrived at — once per feature group, removing that group each
time, and compare against the full model. Using the shipped recipe matters: ablating plain CatBoost
on `log(price)` — the target §12.1 shows is mis-levelled — would measure feature value on a model
already handicapped by something else.

**Scored on validation, not test.** This study *decides* something — which features to keep, and
whether the narrow window earns its cost — and decisions do not read test (§11). The models also
early-stop on validation, which makes the absolute MdAPE here optimistic; that is acceptable
because every variant carries the identical bias and the gate consumes only the **delta** between
variants.

**One group cannot be fully ablated.** The detrended target is `log(price / market level)`, and the
market level is built from `market_median_rolling_3m`. Removing the "market lags" group strips
those columns from the model's *inputs* but cannot remove them from the *label* — the deflator is
computed from the full feature frame so the target stays identical across every run. The market-lag
delta is therefore a **lower bound**: it measures their value as predictors only, not their total
contribution.

**The decision rule** is the 0.15 pp gate from §11. If dropping crime costs less than that, the
81 % data sacrifice is not justified and the next iteration should widen the window rather than
keep squeezing ten years of transactions.

> **Both methodological choices above change the answer, and by enough to reverse the
> recommendation.** Scored on *test* and run on the *plain* target, this same ablation reports crime
> as worth +0.46 pp — comfortably above the gate. On validation and on the shipped recipe, it
> collapses to near a rounding error. Both effects push the same way: the plain-target model is so
> badly mis-levelled (§12.1) that *any* feature carrying market information looks valuable, because
> it is partially compensating for the level error rather than adding signal. Once detrending
> removes that error, most of the apparent value goes with it.

In [ ]:
ablation = ablation_study(SPLITS, VARIANTS, FEATURE_GROUPS, CONFIG, VAL_EVAL)
plot_ablation(ablation)

crime_delta = float(ablation.loc[ablation["Variant"] == "No crime", "MdAPE delta"].iloc[0])
print(f"\nCrime contributes {crime_delta:+.3f} pp of validation MdAPE.")
if crime_delta < ABLATION_GATE_PP:
    print(f"Below the {ABLATION_GATE_PP} pp gate: crime is not worth confining the model to "
          f"2008-2016. The next iteration should drop crime and widen the window to the full "
          f"1995-2024 history (~418k rows, 5x current volume).")
else:
    print(f"Above the {ABLATION_GATE_PP} pp gate: crime earns its place, and the case for "
          f"narrowing the window to keep it is real. Sourcing post-2016 LSOA crime data "
          f"(data.police.uk) would let the window widen without losing the signal.")

ablation

**What this shows.** Validation MdAPE for the full model and for the same recipe with each feature
group removed. A larger positive delta means the group was carrying more weight.

#### Takeaways

| Group | Delta (pp) | Verdict |
|---|---:|---|
| **Prior sales** | **+0.93** | Dominant. Over four times the next-largest group, and the only one comfortably clear of the gate. |
| Market lags | +0.20 | Clears the gate — and is a lower bound, since the deflator keeps them in the label. |
| Crime | +0.11 | **Below the 0.15 pp gate.** |
| Transport | +0.09 | Below the gate. |
| Macro (interest rate) | **−0.05** | Removing it **improves** the model. |

#### Reconciling this with the EDA

Three of the four external drivers §8 presented as important measure as worth roughly nothing here.
That is not a contradiction — the two analyses answer **different questions**:

> §8 asked *"does price vary with this?"* (a marginal association).
> §14.5 asks *"does this add anything the other features do not already carry?"* (a conditional
> contribution).

A feature can score high on the first and zero on the second whenever something else already
carries its information.

| Group | §8 EDA showed | Ablation | Why they differ |
|---|---|---:|---|
| Transport | Steep gradient: £900–950k within 500 m vs. <£350 k beyond 3 km; r = −0.30 | +0.09 pp | **Collinearity.** `borough`, `outcode`, latitude/longitude and `distance_to_center_m` already encode "where". Transport is largely redundant *given* them, despite a strong bivariate signal. |
| Macro | Base rate collapse 5.5 %→0.5 % tracks the price recovery | −0.05 pp | **Already in the target.** `interest_rate` is one value per month shared by every row — a market-level time trend, which is exactly what the detrended target and the market lags absorb. What remains is noise, and removing it helps. |
| Crime | Confound stripped in §8.2 → r ≈ 0.00 | +0.11 pp | **The two agree.** Both routes say the effect is small. |
| Prior sales | §8.3: 61.1 % of rows carry an earlier sale of the same property | +0.93 pp | **The two agree, and it is the biggest thing in the project.** |

#### What this recommends for the next data pull

1. **Crime does not justify the window.** At +0.11 pp it is below the gate, so the 81 % sacrifice
   is not earned. The cells below test whether that verdict survives at finer resolution before it
   is acted on.
2. **Drop `interest_rate`.** A group whose removal *improves* validation MdAPE is not paying for
   its place. It is one column and cheap to keep, but it should not be defended as a signal.
3. **Prior sales are the priority.** The strongest block in the model, and it costs nothing extra —
   it was already in the file.

#### Is that a verdict on crime, or on the resolution it was measured at?

**The distinction §8.2 opened.** The ablation above removed *borough-grain, unnormalised, all-category*
crime. Before the window is widened on the strength of that, the same question gets asked of crime
measured properly.

Below, the winning recipe is refit with **six crime designs** over identical rows and an identical
target — only the crime columns move: borough vs. LSOA grain, an unnormalised count vs. a
population density, and the LSOA total on its own vs. that total alongside its three category
series.

Two details make it a test rather than a formality. Every design is refit under **five seeds**,
because a single gradient-boosted fit cannot separate a 0.1 pp effect from run-to-run variation and
this study decides whether the window survives. And each gain is paired **within** its seed before
averaging, so seed-level variation cancels rather than accumulating.

In [ ]:
CRIME_SEEDS = (42, 43, 44, 45, 46)
crime_designs = crime_resolution_study(SPLITS, VARIANTS, CONFIG, VAL_EVAL, seeds=CRIME_SEEDS)

print()
print(crime_designs.to_string(index=False, formatters={
    "MdAPE mean": "{:.3f}%".format, "MdAPE sd": "{:.3f}".format,
    "Gain vs. no crime": "{:+.3f}pp".format, "Gain sd": "{:.3f}".format}))

In [ ]:
# The same 0.15 pp bar, now applied to the best crime design available rather than to the
# only one that had ever been tried. A gain also has to be large against the seed noise it
# was measured through, or the gate is just reading a number the experiment cannot resolve.
best_crime = crime_designs.loc[crime_designs["Gain vs. no crime"].idxmax()]
noise_floor = crime_designs["MdAPE sd"].mean()

print(f"Best design      : {best_crime['Crime features']}")
print(f"Mean gain        : {best_crime['Gain vs. no crime']:+.3f} pp "
      f"(sd {best_crime['Gain sd']:.3f}, won {int(best_crime['Seeds won'])}"
      f"/{int(best_crime['Seeds'])} seeds)")
print(f"Seed noise floor : {noise_floor:.3f} pp, the mean sd of one design across seeds")
print(f"Gate             : {ABLATION_GATE_PP} pp\n")

clears = best_crime["Gain vs. no crime"] > ABLATION_GATE_PP
beats_noise = best_crime["Gain vs. no crime"] > 2 * noise_floor
if clears and beats_noise:
    print("Crime earns its place at LSOA grain, by a margin the seed spread cannot explain.\n"
          "The case for confining the model to 2008-2016 is real, and widening the window\n"
          "means sourcing post-2016 LSOA crime from data.police.uk rather than dropping it.")
elif clears:
    print("The gain clears the gate but not the seed noise it was measured through.\n"
          "Keeping 2008-2016 -- and discarding 86% of the sale records -- on a difference\n"
          "this experiment cannot resolve would be reading a decision off nothing.")
else:
    print("Crime does not clear the gate at its best resolution, as a count or a rate, whole\n"
          "or split by category. The exploratory and modelling answers agree: the window can\n"
          "widen, and section 18 records what that costs.")

**What this shows.** Each crime design's mean validation MdAPE across five seeds, its standard
deviation, and its paired gain over the no-crime model — then that best gain checked against two
bars.

**Takeaways.**

- **The best design is LSOA grain split by category, at +0.084 pp** — better than the borough-grain
  number, and still **below the 0.15 pp gate**.
- **More importantly, it is below the noise it was measured through.** The seed noise floor — the
  mean standard deviation of one design across seeds — is 0.097 pp, and the best design's own gain
  standard deviation is 0.187 with only 3 of 5 seeds won. The effect is not resolvable by this
  experiment at all.
- **Both bars matter.** A gain that clears the gate but not the noise floor would mean reading a
  decision off a number the experiment cannot measure. Keeping 2008–2016 — and discarding 81 % of
  the sale records — on a difference this size would be exactly that.

**Verdict: crime does not earn the window, at its best available resolution, as a count or a rate,
whole or split by category.** The exploratory answer (§8.2) and the modelling answer agree, by
independent routes. The next iteration should widen to the full 1995–2024 history — roughly 5×
the data — and §18 records what that costs.

This also settles a fairness question: crime was not dismissed on a technicality. It was measured
147× more finely, normalised, and split by category, and still did not clear.

### 14.6 What a property's own history is worth

**The question.** §8.3 established that the file is a repeat-sales panel: 61.1 % of in-window
sales have an earlier sale of the same property. §9 built four prior-sale features on that
basis. This section is where those features had to justify themselves — **the study that put
them in `FEATURES`, run before they were adopted.**

The candidate set was six columns; four are carried today. This measures all of them.

**Causally clean, but "strictly before" is load-bearing.** These features read only sales strictly
prior to the row being predicted. Three things could break that, and each is handled: the 102,527
duplicate rows are removed first (or an as-of merge can return the very row it is meant to
predict); same-day conflicting prices are collapsed to a median; and `assert_no_lookahead` is a hard
check in §9, with a matching assertion in the §17 self-checks.

**Same protocol as the crime study above** — same 0.15 pp gate, seeds paired within run, so the two
results are directly comparable.

In [ ]:
prior_sale_designs = prior_sale_study(SPLITS, VARIANTS, CONFIG, VAL_EVAL, seeds=(42, 43, 44))

print()
print(prior_sale_designs.to_string(index=False, formatters={
    "MdAPE mean": "{:.3f}%".format, "MdAPE sd": "{:.3f}".format,
    "Gain vs. no prior sale": "{:+.3f}pp".format, "Gain sd": "{:.3f}".format}))

In [ ]:
best_prior = prior_sale_designs.loc[prior_sale_designs["Gain vs. no prior sale"].idxmax()]
print(f"\nBest design : {best_prior['Prior-sale features']}")
print(f"Mean gain   : {best_prior['Gain vs. no prior sale']:+.3f} pp "
      f"(sd {best_prior['Gain sd']:.3f}), won {int(best_prior['Seeds won'])}"
      f"/{int(best_prior['Seeds'])} seeds")
print(f"Gate        : {ABLATION_GATE_PP} pp\n")

if best_prior["Gain vs. no prior sale"] > ABLATION_GATE_PP:
    print("Comfortably clear, and on every seed. For scale, the same protocol scores crime at\n"
          "its best resolution (reported above) at just +0.08 pp, with a standard deviation\n"
          "larger than its mean.\n"
          "These four columns are therefore carried in FEATURES; the two dropped from the\n"
          "candidate set -- log_prev_sale_price and n_prior_sales -- earned nothing, since a\n"
          "tree splits on order and the log of a column it already holds is the same ordering.")
else:
    print("Below the gate: the prior-sale group does not justify its complexity.")

**What this shows.** Each candidate prior-sale feature set, its mean validation MdAPE across seeds,
and its paired gain over carrying no prior-sale features at all.

**Takeaways.**

- **The group clears the gate by roughly 6×, and wins on every seed** — the opposite profile to
  crime, which cleared neither the gate nor its own noise floor. For scale: crime's best design
  scored +0.084 pp with a standard deviation larger than its mean; this scores +0.913 pp with all
  seeds agreeing.
- **Four columns were adopted, two rejected.** `prev_sale_price`, `has_prev_sale`,
  `years_since_prev_sale` and `prev_sale_days_since_start` are in `FEATURES`. Carrying all six
  scored *worse* than carrying four, so `log_prev_sale_price` and `n_prior_sales` earn nothing — a
  tree splits on order, and the log of a column it already holds is the same ordering.
- **The elapsed-time pair doubles the value of the price alone.** Prior price by itself is worth
  about +0.45 pp; adding *when* that sale happened takes it to +0.91 pp. That is the substantive
  finding: **a past price is only interpretable against how long ago it was paid.** £400k in 2009
  and £400k in 2015 say very different things about what a property is worth today.
- **This is the single largest modelling gain in the project** — larger than any architecture
  choice, and second only to the target reframing in §12.1.

**The cost of getting here.** These features were sitting in the source file from the beginning.
The pipeline read a price *history* as a transaction log for its entire first version, which is a
reminder that the largest wins often come from re-reading the data you already have rather than
from adding models on top of it.

---
## 15. Conformal safety bound and the flip scanner

**What happens here.** The point estimate becomes an actionable floor.

A predicted price is not enough to justify spending money. What an investor needs is a **floor**: a
value the property is very unlikely to be worth less than.

**Multiplicative split conformal prediction** supplies one. On a dedicated **calibration split** —
separate from the validation set used for early stopping and model selection — compute the ratio of
actual to predicted price for every property:

$$r_i = \frac{y_i}{\hat{y}_i}$$

and take the 10th percentile, $q_{10}$. For a new property the floor is $\hat{y} \times q_{10}$, and
by construction roughly 90 % of properties should sit above it.

**Why ratios rather than differences.** Absolute residuals in this market are heteroscedastic:
errors widen sharply as price rises. A £3 M house misses by far more pounds than a £200k flat
while being no less accurate in percentage terms, so a single absolute quantile would be far too
loose at the bottom of the market and far too tight at the top. The ratio form scales the buffer with the price,
so one calibration serves both tiers.

**Why calibration gets its own split, not the validation set.** Split conformal assumes the
calibration residuals were not used to fit the model. Validation *was* used — for early stopping and
architecture choice — so a model's iteration count is implicitly tuned to minimise error on exactly
those rows, and calibrating there would make $q_{10}$ optimistically tight. The calibration split
sits strictly between validation and test in time and is untouched by anything except this cell.

**On the exchangeability assumption.** Classical conformal prediction assumes calibration and test
data are exchangeable, which a chronological split does not strictly guarantee — the market drifts.
The empirical coverage check below is therefore **not a formality**; it is the actual evidence that
the guarantee holds, and it is reported honestly whether or not it lands on target.

A property is flagged as a **flip candidate** when its transaction price falls below its floor.
Calibration and scanning both call `bundle.predict`, so the model being calibrated is provably the
model being deployed — `BEST` from §14, chosen on validation MdAPE alone.

In [ ]:
# Calibrated on CALIB_EVAL, not VAL_EVAL: the model's hyperparameters (early stopping,
# architecture choice) were already tuned against validation, so computing q_10 there would
# calibrate the safety bound on data the model has effectively already seen.
Q_SAFETY = calibrate_conformal(BEST, CALIB_EVAL, SPLITS, CONFIG.conformal_alpha)
scan = scan_for_flips(BEST, TEST_EVAL, SPLITS, Q_SAFETY)
flips = scan[scan["is_flip"]].sort_values("margin", ascending=False)

coverage = (scan["actual_price"] >= scan["safe_lower_bound"]).mean() * 100
target = (1 - CONFIG.conformal_alpha) * 100
print("\n--- Empirical coverage on the held-out test set ---")
print(f"Target confidence : {target:.2f}%")
print(f"Actual coverage   : {coverage:.2f}%   ({coverage - target:+.2f} pp)")
print(f"Properties scanned: {len(scan):,}")
print(f"Flip candidates   : {len(flips):,} ({len(flips) / len(scan) * 100:.2f}%)")
if len(flips):
    print(f"Median margin     : \N{POUND SIGN}{flips['margin'].median():,.0f}")
flips.head(10)

**What this shows.** The calibrated multiplier, empirical coverage on the held-out test set against
the 90 % target, how many properties were scanned, how many were flagged, and the median margin
below the floor.

**How to read it.**

- **Coverage is the number that validates the method.** If empirical coverage lands close to 90 %,
  the exchangeability assumption survived the chronological split well enough to trust the bound. A
  coverage *below* target means the floor is too tight — it flags more candidates than it should,
  and some of them are not genuinely underpriced.
- **A shortfall of a point or so is expected, not alarming.** The market drifted between the
  calibration window and the test window, which is exactly the departure from exchangeability
  flagged above. It is reported rather than corrected because correcting it against test would
  turn test into a tuning signal (§11).
- **The flip rate is a screening rate, not a hit rate.** A flagged property sold below a calibrated
  valuation floor. It says nothing about *why* — and §18 is explicit that properties are usually
  cheap for reasons this dataset does not record.
- **Margin is the buffer in pounds**, not expected profit. Turning it into profit means subtracting
  stamp duty, refurbishment, financing and fees — improvement #3 in §18.

In [ ]:
plot_flip_margins(flips, scan, Q_SAFETY, CONFIG)

**What this shows.** Left: the distribution of margins below the floor across flagged properties.
Right: actual price against predicted value for every scanned property, with the floor line drawn
through it.

**Takeaways.**

- **Points below the floor line are the candidates.** The line is not a decision boundary the model
  learned — it is the calibrated $\hat{y} \times q_{10}$ bound, applied uniformly.
- **The margin distribution is right-skewed**: most flagged properties sit just below the floor, a
  few sit far below. The far tail is where the screen is most interesting and also where it is most
  likely to be picking up something the data does not record — a short lease, a structural problem,
  a distressed sale.
- **The floor is a single global ratio**, so it is loose for easy-to-value properties and tight for
  hard ones. Conditional (Mondrian) conformal prediction — improvement #4 in §18 — is the fix, and
  Romano et al.'s conformalized quantile regression (see the [appendix](#appendix-related-work)) is
  the standard route to it.

---
## 16. Persisting the run

**What happens here.** A model that exists only inside a kernel session is not a deliverable. This
cell writes the selected estimators, the conformal multiplier, the full leaderboard and a
machine-readable manifest to `artifacts/`.

| Artifact | Contents |
|---|---|
| `manifest.json` | selected model, feature list, target, conformal α and multiplier, config, and both leaderboards |
| `leaderboard.csv` | every model × split scored during the run |
| `model.joblib` | the fitted estimators, the pinned category dtypes and the target encoder |

**Note on scope.** The `predict` closures built in §12 are not picklable by design, so what is
persisted is the underlying estimators plus everything needed to rebuild the closure. Reloading
means importing `lff` and calling the matching trainer's predict logic against the saved
artefacts — the *package* is the deployment unit, and this notebook is the narrative that produced
it. The remaining gap is a retraining job that fails loudly when the §17 self-checks do (§18, #8).

In [ ]:
persist_run(BEST, Q_SAFETY, CONFIG, RESULTS, SPLITS)

---
## 17. Automated self-checks

**What happens here.** A notebook with no assertions is a notebook that fails silently. These
checks encode the invariants the pipeline depends on, and **each one guards a defect that was
actually present in an earlier version**:

1. **Chronological integrity** — no training row may be dated after a validation row.
2. **No dead feature columns** — the bug that left four categorical columns entirely NaN would have
   been caught here immediately.
3. **Categorical levels are pinned from training**, so a category code means the same thing at fit
   time and predict time.
4. **Lagged market features never see the present** — a month's own median must not equal its own
   predictor.
5. **Prior sales strictly precede the row that reads them** — asserted, not merely reported,
   because a single violation is target leakage rather than a quality issue.
6. **Conformal coverage** lands near its nominal level — asserted on a *held-out slice of the
   calibration split*, never on test. The multiplier is refitted on the first 70 % of calibration
   and scored on the remaining 30 %, because checking coverage on the same rows the multiplier was
   fitted on is an identity, not a test. Test-set coverage is printed alongside for information and
   deliberately carries **no** assertion: making a run fail on a held-out metric turns that metric
   into a tuning signal.
7. **Re-runnability** — the raw station table still exists and is untouched, which an earlier
   version's variable clobbering broke.

Run them after any change to the pipeline. The same probes run in `tests/` against a committed
500-row fixture, in about two seconds.

In [ ]:
run_self_checks(SPLITS, CONFIG, BEST, coverage, target, RAW, df_master)

**What a green run proves — and what it does not.**

It proves the structural invariants hold: no temporal leakage across the splits, no prior sale read
from the future, no dead columns, encoders pinned to training, and a conformal bound that covers at
roughly its nominal rate on data it was not fitted on.

It does **not** prove the model is good, that the features are the right ones, or that the
protocol was followed while the notebook was being *written* — that last one is what §18 exists to
disclose.

---
## 18. Limitations and where to take this next

### What this model is not

* **The scanner evaluates completed sales, not properties you can buy.** `TARGET` is
  `history_price` — what a property *actually sold for*. A property that sold below its floor in
  2016 validates the valuation model; it is not a listing anyone can act on today. A live scanner
  needs a listings feed (asking prices) and a calibration step for the asking-to-sold gap, which is
  a different distribution and not in this dataset.
* **The "flip candidate" flag is a statistical claim, not a financial one.** It says the price is
  below a calibrated valuation floor. It says nothing about stamp duty, refurbishment, holding
  costs, agent fees, or *why* the property is cheap — and properties are usually cheap for a reason
  the data does not record — a lease with few years left to run, structural problems, a seller who
  needs out quickly. Treat the margin as a screening signal, not expected profit.
* **Conformal coverage is marginal, not conditional.** Roughly 90 % of properties sit above the
  floor *overall*. That does not guarantee 90 % inside any particular slice — not within a single
  expensive borough such as Kensington, and not within the top price decile.
* **The 2008–2016 window ends a decade ago**, and four shocks that moved this market fall
  entirely outside it: the 2016 Brexit referendum, which hit London prices hardest of anywhere in
  the UK; the 3 % stamp-duty surcharge on additional properties introduced the same year, aimed
  squarely at buy-to-let and flipping; the pandemic, which inverted the usual premium on being
  central; and the 2022 interest-rate cycle, which took the base rate from near zero to over 5 %
  and repriced every mortgage in the country. Nothing here should be pointed at today's market
  without recalibration.
* **Gain-based feature importance is not causal**, and this feature set is heavily collinear
  (latitude, longitude, `borough`, `outcode` and distance-to-centre all encode "where").
* **The evaluation universe is defined using the target.** Two filters decide which rows are
  eligible to be scored, and both read `price`: the £1,500/sqm floor (§7, applied before the split
  to all four slices) and the £4 M cap (§10). Both are defensible as definitions of "the standard
  market this product serves", and neither is a temporal leak — but neither is identifiable at
  prediction time either. The reported MdAPE therefore describes a population you could not
  actually select in production, where price is the unknown.

### Resolved since the first version

* **A property's own transaction history is now used.** The file is a repeat-sales panel — 314,895
  distinct sales across 137,760 addresses, 61.1 % of in-window sales having an earlier sale — and
  the first version used none of it. Four prior-sale features are now in `FEATURES`, worth
  **+0.91 pp** of validation MdAPE (§14.6): the largest feature-side gain in the project. *Still
  open:* a proper repeat-sales index, and the group-aware split below.
* **Crime has been tested at LSOA resolution.** Borough grain was a 147× aggregation loss, and the
  original verdict on crime was arguably a verdict on that aggregation. §8.2 and §14.5 now measure
  it at native LSOA grain, as a rate as well as a count, split by category — **it still does not
  clear the gate**, and the finding is now about crime rather than about resolution.
* **The pipeline has been lifted out of the notebook.** §1–§17 live in `src/lff/`, with a `pytest`
  suite over a committed 500-row fixture that runs the leakage probes in ~2 seconds rather than only
  at the end of a full run, and `nbstripout` in a pre-commit hook so outputs never land in git.

### Ranked improvements

**1 — Split by property, not only by time.** §14.1 quantifies the overlap. Add a group-aware split
keyed on `fullAddress` and report both numbers; if the gap is large, the headline metric is
measuring memorisation as much as valuation. This matters more now that prior-sale features exist,
since they act precisely on repeat properties.

**2 — Widen the window to 1995–2024.** §14.5 settles that crime does not justify the 81 % of the
record it costs. Dropping it takes the dataset from ~60k to ~315k sales — roughly 5× — which is
almost certainly worth more than every remaining item on this list combined.

**3 — Walk-forward backtesting.** One 60/15/10/15 cut yields one number with no error bar. Rolling
origin evaluation (expanding window, refit each year) gives a distribution of MdAPE and reveals
whether performance depends on which slice of the cycle you happened to test on.

**4 — Make the margin an actual P&L.** Add the costs a buyer actually pays: stamp duty (the UK's
banded purchase tax, plus the 3 % surcharge that applies to exactly this kind of second-property
purchase), refurbishment, financing at the prevailing rate, and agent and legal fees. `margin`
becomes expected profit, and the scanner can rank by return rather than by pounds.

**5 — Conditional (Mondrian) conformal prediction.** Calibrate separate multipliers per borough and
per price decile so coverage holds *within* the segments an investor actually shops in.

**6 — Drop `interest_rate`, and revisit the remaining thin groups.** §14.5 measures its removal as
an *improvement*. Transport at +0.09 pp is similarly marginal given how much else encodes location —
though travel *time* to Zone 1, rather than straight-line distance, might not be.

**7 — Features the data does not yet carry.** Lease length above all. A leasehold flat with 70
years left is worth materially less than an otherwise identical one with 950, and extending a
short lease costs tens of thousands — so this single missing column is a plausible part of what the
scanner currently mistakes for underpricing.

**8 — Interpretability that survives collinearity.** SHAP values on the winning model, plus
permutation importance grouped over the location block, so "where" is credited once rather than
split five ways.

**9 — Quantile regression as a conformal alternative.** Fitting the 10th percentile directly
(`objective='reg:quantileerror'`) gives a floor that adapts per property, where the current
multiplicative bound applies one global ratio to everything.

**10 — Operational hardening.** *Partly done* (see above). Still outstanding: a retraining job that
fails loudly when the §17 self-checks do.

---
## Appendix: checking against real, current listings

**What this is.** `scripts/live_flip_scan/` is a first, deliberately uncalibrated attempt at the
live scanner §18 says this dataset can't build on its own: it scrapes current Rightmove listings
(robots.txt checked at runtime, rate-limited, every response cached), substitutes five public data
sources for the features that only exist inside this project's own 2008–2016 corpus, retrains the
selected model in-process, and scores the live listings through the same conformal scanner as
§15. It runs outside the notebook — scraping plus a full retrain takes the better part of an hour
— so this cell is a static record of the last run rather than something re-executed top-to-bottom.

| Live-data proxy | Real feature it stands in for |
|---|---|
| UK HPI (`landregistry.data.gov.uk`) | `market_median_rolling_3m/12m`, `lagged_borough_median_sqm` |
| `data.police.uk` | `crime_volume`, `crime_volume_prev_12m` (point-radius, not borough-sum; the 12-month figure is the latest month annualised, not a real trailing sum) |
| EPC register | `floorAreaSqM`, `currentEnergyRating` fallback (unused this run — no API key configured) |
| HM Land Registry Price Paid Data | `prev_sale_price` and the rest of §14.6's block, matched on postcode + street |
| `lff.spatial`, reused directly | borough, distance to nearest station/centre |

**Result, 200 current London listings (2026-08-23).**

| | Flip rate | N |
|---|---|---|
| Held-out test set (§15, completed sales) | 13.39 % | 8,859 |
| Live Rightmove listings (asking prices) | 29.00 % | 200 |

**Read this carefully.** The live figure is *higher*, not lower — the opposite of what the
asking-vs-sold gap alone would predict, since an asking price sitting above the eventual sold
price should make fewer listings look underpriced, not more. Splitting the predicted/asking ratio
by borough points at why this probably isn't a real signal: a handful of boroughs (Camden,
Kensington & Chelsea) show predictions 1.7–2.5× the asking price, more consistent with an
artifact of the UK HPI market-level proxy or the 10–18 year extrapolation on `days_since_start`
than genuine mispricing. Treat 29.00 % as inconclusive, not as a finding about the 2026 market —
the script prints the full proxy-vs-real breakdown with every run.

```bash
python scripts/run_live_flip_scan.py --stage all
```


---
## Appendix: related work

None of the ingredients here are new on their own — hedonic valuation, repeat-sales history, and
distribution-free prediction intervals each have a literature. What is specific to this project is
the combination: a hedonic model of London stock in 2008–2016, given the property's own sale
history, wrapped in a *one-sided* calibrated floor so the output is a screen a buyer can act on
rather than a point estimate.

| Work | What it is | Where it touches this project |
|---|---|---|
| Rosen (1974), [*Hedonic Prices and Implicit Markets*](https://www.journals.uchicago.edu/doi/10.1086/260169), JPE 82(1):34–55 | The theory that prices a differentiated good as a bundle of measured attributes, each carrying an implicit price. | The framing of §5–§9. Every feature — floor area, tenure, distance to a station, borough — is an attribute whose implicit price the model is estimating; the tree models just drop the linear-in-attributes assumption. |
| Bailey, Muth & Nourse (1963), [*A Regression Method for Real Estate Price Index Construction*](https://www.semanticscholar.org/paper/A-Regression-Method-for-Real-Estate-Price-Index-Bailey-Muth/8384788b906b9cbde02c20fede181f7163fc29eb), JASA 58:933–942; extended by [Case & Shiller (1987)](https://www.nber.org/system/files/working_papers/w2506/w2506.pdf) | Repeat sales: use properties sold more than once to separate market movement from property quality, since the property is held fixed between the two sales. | Both halves of that idea are used here, in opposite directions. §12.1 divides out the market level to get a stationary target; §8.3 and §14.6 go the other way and feed the *previous sale price* back in as a feature — the single strongest signal in the data (+0.91 pp MdAPE). §14.1 also reports errors on repeat properties separately. |
| Gibbons (2004), [*The Costs of Urban Property Crime*](https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1468-0297.2004.00254.x), Economic Journal 114(499):F441–F463 | A hedonic study of London specifically: criminal damage capitalises into prices (≈1 % per tenth of a standard deviation in Inner London), burglary does not. | The closest prior work to §8.2 and §14.5, and it predicts what we find — a real but small effect that is category-dependent. Worth reading as the reason the crime block earns so little once location is already in the model, rather than as a contradiction of it. |
| Lei, G'Sell, Rinaldo, Tibshirani & Wasserman (2018), [*Distribution-Free Predictive Inference for Regression*](https://arxiv.org/abs/1604.04173), JASA 113(523):1094–1111 | The reference treatment of split conformal prediction: finite-sample marginal coverage on top of any regressor, with no distributional assumptions. | §15 is split conformal with a *ratio* nonconformity score (actual/predicted) and only the lower tail kept, because a flip screen cares about the floor and not the ceiling. |
| Romano, Patterson & Candès (2019), [*Conformalized Quantile Regression*](https://papers.nips.cc/paper/8613-conformalized-quantile-regression), NeurIPS 32:3538–3548 | Conformal intervals that adapt their width to the input, rather than one global correction. | The obvious next step for §15. The multiplier *q* here is a single constant across the whole market, so coverage holds on average but the floor is loose for easy properties and tight for hard ones. [MAPIE](https://github.com/scikit-learn-contrib/MAPIE) is the usual scikit-learn implementation of both this and the split method above. |
| [Zillow Prize](https://www.zillow.com/z/info/zillow-prize/) (Kaggle, 2017–2019) | The largest public competition on automated valuation: 3,800+ teams predicting the Zestimate's log error; the winners improved on the benchmark by ~13 %, and Zillow reports the national median error falling from ~4.5 % to under 4 %. | The practitioner reference point for §12–§14 — gradient-boosted ensembles over property, location and time features are what won there too. The error rates are *not* comparable to the MdAPE here: a different market, no listing or interior data, and every transaction scored rather than on-market homes only. |